In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from scipy.interpolate import interpn

from thermal_loads_260924 import *

In [ ]:
# environment settings: 
pd.set_option('display.max_column',None)
pd.set_option('display.max_rows',50)
pd.set_option('display.max_seq_items',None)
pd.set_option('display.max_colwidth', 500)
pd.set_option('expand_frame_repr', True)

np._core.arrayprint.set_printoptions(linewidth= 180)
np.set_printoptions(threshold=np.inf)

# Data

In [ ]:
project = '260924_House'

#### Construction quality level (thermal bridges)
'A' : New building with certified minimization of thermal bridges  
'B' : New building compliant with general codes of practice  
'C' : Building with internal insulation interrupted by solid floor slabs  
'D' : Other buildings  

In [ ]:
building_quality = 'B'

#### Heating system (stratification)   
'HAC' : Hot-air heating without additional destratification   
'HAD' : Hot-air heating with additional destratification   
'CRP' : Ceiling-mounted radiant panels   
'RTH' : Radiant tube heating   
'IRH' : Infrared radiant heater   
'SUR' : Integrated surface heat emission   
'RAD' : Radiators   
'NOT' : No heating system specified

In [ ]:
heating_syst   = 'RAD'

#### Postal code

In [ ]:
CP = 4950 # Base external temperatures

#### Internal temperature

In [ ]:
t_in  = 20  # Base internal temperature 

#### Infiltration rate at 50 Pa

In [ ]:
n_50  = 0.6

#### Windows and doors U-values

In [ ]:
U_wd  = 1.50  #[W/m2/K]
U_dr  = 1.67   #[W/m2/K]

In [ ]:
col_names = [ 'building_quality', 'heating_syst', 'CP', 't_in', 'n_50', 'U_wd', 'U_dr']
gen_data  = [ building_quality, heating_syst, CP, t_in, n_50, U_wd, U_dr]

df_gen_data = pd.DataFrame({'Names': col_names, 'Values': gen_data})
df_gen_data = df_gen_data.set_index('Names')

#### Additional Power for Reheating

In [ ]:
# Choose between EN12831 Annex F Tables 'F1' or 'F3'
rh_table = 'F1'

# 'L' : Lightweight structure, 'H' : Heavy structure
th_mass = 'L'

# Infiltration rate during setback time (min 0.1 vol/h, max 0.5 vol/h)
n_inf_setback = 0.5

# Heating set back duration (max 168 h for table 'F1')
time_setback_h = 8

# Reheat time in hours (min 0.5 h)
time_reheat_h = 2

# Minimum reheat factor (W/m² of floor heated area)
f_rh_min = 10 # W/m²

In [ ]:
col_names = ['rh_table', 'thermal_mass', 'infiltration_rate_setback', \
             'time_setback_h', 'time_reheat_h', 'f_rh_min']
rh_data = [rh_table, th_mass, n_inf_setback, time_setback_h, time_reheat_h, f_rh_min]

df_rh_data = pd.DataFrame({'Names': col_names, 'Values': rh_data})
df_rh_data = df_rh_data.set_index('Names')

#### Ventilation

In [ ]:
# Floor heated areas
A_fl_office   =     8.30 * 5.60 
A_fl_dwelling = 2 * 5.60 * 5.60 

# Entity, floor heated area, ceiling height, fresh air flow (m³/h), efficiency of heat recovery
vent = \
[('office',   A_fl_office,   2.51,  250, 0.86),
 ('dwelling', A_fl_dwelling, 2.785, 250, 0.86)]

dfv = pd.DataFrame(vent, columns=('zone', 'area_fl', 'h_ceiling', 'q_su_m3h', 'epsilon_rec'))
dfv = dfv.set_index('zone')

#### Materials

In [ ]:
materials_list = []

materials_list.append({"material":'Plaster_Panel'      ,"lambda":0.32, "rho":1200,"c":1000})
materials_list.append({"material":'Dry_screed_Panel'   ,"lambda":0.38, "rho":1225,"c":1000})
materials_list.append({"material":'DF_panel'           ,"lambda":0.20, "rho":1200,"c":1000})
materials_list.append({"material":'Glass_wool'         ,"lambda":0.035,"rho":20,  "c":840})
materials_list.append({"material":'OSB'                ,"lambda":0.12, "rho":553, "c":1700})
materials_list.append({"material":'Wood_cement_Panel'  ,"lambda":0.35, "rho":1250,"c":1000})
materials_list.append({"material":'Wood'               ,"lambda":0.13, "rho":600, "c":1600})
materials_list.append({"material":'PIR'                ,"lambda":0.025,"rho":40,  "c":840})
materials_list.append({"material":'EPS_board'          ,"lambda":0.034,"rho":25,  "c":1300})
materials_list.append({"material":'Expanded_foam'      ,"lambda":0.036,"rho":12,  "c":1300})
materials_list.append({"material":'Waterproof_membrane',"lambda":0.230,"rho":1300,"c":1000})
materials_list.append({"material":'Gravels'            ,"lambda":2.0,  "rho":2000,"c":1000})
materials_list.append({"material":'Air_layer_160mm'    ,"lambda":0.711,"rho":1.2, "c":1000})

materials = pd.DataFrame(materials_list)
# materials

In [ ]:
# Add a composite material
# Data = material list, names of the first and second materials, widths of the first and second materials
composite_material(materials_list, 'Insulation_layer', 'Glass_wool', 'Wood', 0.6, 0.06)

materials = pd.DataFrame(materials_list)
materials = materials.set_index('material')

#### Wall types layers

In [ ]:
# Wall layers from indoor to outdoor

wt_layers = \
[('floor_ext', 0, 0.015, 'Wood'),
 ('floor_ext', 1, 0.028, 'Dry_screed_Panel'),
 ('floor_ext', 2, 0.018, 'OSB'),
 ('floor_ext', 3, 0.300, 'Insulation_layer'),
 ('floor_ext', 4, 0.015, 'Wood_cement_Panel'),
 ('floor_ext', 5, 0.100, 'PIR'),
 
 ('floor_int', 0, 0.015, 'Wood'),
 ('floor_int', 1, 0.028, 'Dry_screed_Panel'),
 ('floor_int', 2, 0.018, 'OSB'),
 ('floor_int', 3, 0.160, 'Air_layer_160mm'),
 ('floor_int', 4, 0.140, 'Insulation_layer'),
 ('floor_int', 5, 0.015, 'Wood_cement_Panel'),
 ('floor_int', 6, 0.080, 'Insulation_layer'),
 ('floor_int', 7, 0.060, 'Gravels'),
 ('floor_int', 8, 0.030, 'DF_panel'),
 
 ('roof', 0, 0.030, 'DF_panel'),
 ('roof', 1, 0.060, 'Gravels'),
 ('roof', 2, 0.080, 'Insulation_layer'),
 ('roof', 3, 0.015, 'OSB'),
 ('roof', 4, 0.300, 'Insulation_layer'),
 ('roof', 5, 0.018, 'OSB'),
 ('roof', 6, 0.120, 'PIR'),
 ('roof', 7, 0.005, 'Waterproof_membrane'),

 ('terrace', 0, 0.030, 'DF_panel'),
 ('terrace', 1, 0.060, 'Gravels'),
 ('terrace', 2, 0.080, 'Insulation_layer'),
 ('terrace', 3, 0.015, 'OSB'),
 ('terrace', 4, 0.300, 'Insulation_layer'),
 ('terrace', 5, 0.018, 'OSB'),
 ('terrace', 6, 0.005, 'Waterproof_membrane'),

 ('wall_ext', 0, 0.015, 'Plaster_Panel'),
 ('wall_ext', 1, 0.090, 'Insulation_layer'),
 ('wall_ext', 2, 0.018, 'OSB'),
 ('wall_ext', 3, 0.300, 'Insulation_layer'),
 ('wall_ext', 4, 0.016, 'Wood_cement_Panel') 
]

wt_layers = pd.DataFrame(wt_layers, columns=('wall_type', 'layer', 'thickness', 'material'))
wt_layers = wt_layers.set_index('wall_type')

# List of wall types
wt_layers.index.unique().tolist()

#### Wall types environments

In [ ]:
# list of wall types in contact with the ground
GRwalls = []

# list of wall types with thermal bridges
TBwalls = ['floor_ext', 'roof', 'terrace', 'wall_ext']

# list of wall types adjacent to another building entity
BEwalls = ['floor_int']

wt_env = pd.DataFrame({'wall_type': wt_layers.index.unique().tolist()})
wt_env['ground_contact']  = wt_env['wall_type'].isin(GRwalls)
wt_env['thermal_bridges'] = wt_env['wall_type'].isin(TBwalls)
wt_env['adjacent_entity'] = wt_env['wall_type'].isin(BEwalls)

wt_env = wt_env.set_index('wall_type')

#### Wall dimensions and environments

In [ ]:
# List of wall types
wt_layers.index.unique().tolist()

In [ ]:
# Wall thicknesses
dftk = (
    wt_layers.groupby(level='wall_type')['thickness']
             .sum()
             .round(2)
             .reset_index()
        )
thk = dict(zip(dftk['wall_type'], dftk['thickness']))

In [ ]:
twle = thk['wall_ext']
tfli = thk['floor_int']
tfle = thk['floor_ext']
trf  = thk['roof']

Ho = dfv.loc['office', 'h_ceiling']
Hd = dfv.loc['dwelling', 'h_ceiling']

hwo =     Ho + tfle + tfli/2
hwd = 2 * Hd + trf  + tfli + tfli/2

# Zone, azimuth, slope, wall_type, horiz dim1, horiz dim2, vert dim, h_in, h_out, temp environnement
walls = \
[('office',     0,   0, 'floor_ext', 8.30 + 2 * twle, 5.60 + 2 * twle,   0,  8,  8, 't_ext'),
 ('office',   180,  90, 'wall_ext',  5.60 + 2 * twle, 0,               hwo,  8,  8, 't_ext'),
 ('office',   -90,  90, 'wall_ext',  8.30 + 2 * twle, 0,               hwo,  8,  8, 't_ext'),
 ('office',     0,  90, 'wall_ext',  5.60 + 2 * twle, 0,               hwo,  8,  8, 't_ext'),
 ('office',    90,  90, 'wall_ext',  8.30 + 2 * twle, 0,               hwo,  8,  8, 't_ext'),
 ('office',     0, 180, 'terrace',   5.60 + 2 * twle, 8.30 - 5.60,       0,  8,  8, 't_ext'),
 ('office',     0,   0, 'floor_int', 5.60 + 2 * twle, 5.60 + 2 * twle,   0,  8,  8, 't_avg'),
 
 ('dwelling',   0,   0, 'floor_int', 5.60 + 2 * twle, 5.60 + 2 * twle,   0,  8,  8, 't_avg'),  
 ('dwelling', 180,  90, 'wall_ext',  5.60 + 2 * twle, 0,               hwd,  8,  8, 't_ext'),
 ('dwelling', -90,  90, 'wall_ext',  5.60 + 2 * twle, 0,               hwd,  8,  8, 't_ext'),
 ('dwelling',   0,  90, 'wall_ext',  5.60 + 2 * twle, 0,               hwd,  8,  8, 't_ext'),
 ('dwelling',  90,  90, 'wall_ext',  5.60 + 2 * twle, 0,               hwd,  8,  8, 't_ext'),
 ('dwelling',   0, 180, 'roof',      5.60 + 2 * twle, 5.60 + 2 * twle,   0,  8, 23, 't_ext')]

wl = pd.DataFrame(walls, columns=('zone', 'azimuth', 'slope', 'wall_type', \
                                  'dimh1', 'dimh2', 'dimv', 'h_in', 'h_out', 't_env_name')).set_index('zone')
# wl

#### Windows

In [ ]:
# Zone, azimuth, slope, wall_type, number of identical windows, width, height, sill height

windows = \
[('office',     0, 90, 'wall_ext', 1, 0.90, 2.59, 0.00),
 ('office',   -90, 90, 'wall_ext', 2, 0.90, 2.59, 0.00),
 
 ('dwelling', -90, 90, 'wall_ext', 1, 1.46, 1.78, 0.70),
 ('dwelling',  90, 90, 'wall_ext', 1, 0.56, 0.84, 0.90),
 ('dwelling',   0, 90, 'wall_ext', 5, 1.20, 2.59, 0.00),
 ('dwelling',   0, 90, 'wall_ext', 1, 0.61, 2.59, 0.00),
 ('dwelling', -90, 90, 'wall_ext', 3, 0.90, 2.59, 0.00),
 ('dwelling', -90, 90, 'wall_ext', 1, 0.90, 2.59, 0.00),
 ('dwelling', -90, 90, 'wall_ext', 3, 0.90, 2.59, 0.00)]

wd = pd.DataFrame(windows, columns=('zone', 'azimuth', 'slope', 'wall_type', \
                                    'number', 'width', 'height', 'sill')).set_index('zone')
# wd

#### Doors

In [ ]:
# Zone, azimuth, slope, wall_type, number of identical external doors, width, height

doors = \
[('office',   180, 90, 'wall_ext', 1, 1.00, 2.17),
 ('dwelling', 180, 90, 'wall_ext', 1, 1.00, 2.17)]

dr = pd.DataFrame(doors, columns=('zone', 'azimuth', 'slope', 'wall_type', \
                                  'number', 'width', 'height')).set_index('zone')
# dr

# Compute Heat Losses

In [ ]:
# dfwt  : Walls thicknesses [m] and conductive resistance [m²K/W]
# dfAU  : Walls U_values [W/m²K] - Temperature Factors - Areas [m²] - Heat Tranfer Coefficients [W/K]
# dfgrAU : Walls Areas [m²] and Heat Tranfer Coefficients [W/K] by zone and wall type
# dfAHT : Wall Areas [m²] and Heat Tranfer Coefficients [W/K] by zone
# dfH   : Total Heat Tranfer Coefficients by zone [W/K]
# dfPHI : Emission and production Heat losses and Loads by zone [W]
# dffv  : Emission Heat losses and loads by zone and by facade [W]
# dffPHI : Emission Heat losses and loads by facade [W]

dfwt, dfAU, dfgrAU, dfAHT, dfH, dfPHI, dffv, dffPHI = thermal_loads(gen_data, rh_data, dfv, \
                  materials, wt_layers, wt_env, wl, wd, dr)

In [ ]:
# Display style for the dataframes
styles = [dict(selector="caption",
                       props=[("text-align", "middle"),
                              ("font-size", "110%"),
                              ("font-weight", "bold"),
                              ("color", 'black'),
                              ("padding-bottom", "10px"),
                              ("white-space", "nowrap")
                             ])]

In [ ]:
numeric_cols = dfAU.select_dtypes(include='number').columns

dfAU.style \
    .format("{:.2f}", subset=numeric_cols) \
    .set_caption("Walls U_values [W/m²K] - Temperature Factors - Areas [m²] - Heat Tranfer Coefficients [W/K]") \
    .set_table_styles(styles)

In [ ]:
numeric_cols = dfgrAU.select_dtypes(include='number').columns

dfgrAU.style \
    .format("{:.2f}", subset=numeric_cols) \
    .set_caption("Walls Areas [m²] and Heat Tranfer Coefficients [W/K] by zone and wall type") \
    .set_table_styles(styles)

In [ ]:
bold_cols = ['area', 'H_T']
bold_cols = [col for col in dfAHT.columns if col.startswith(tuple([x + ' ' for x in bold_cols]))]

dfAHT.style \
    .format("{:.2f}") \
    .set_properties(subset=bold_cols, **{"font-weight": "bold"}) \
    .set_caption("Wall Areas [m²] and Heat Tranfer Coefficients [W/K] by zone") \
    .set_table_styles(styles)

In [ ]:
bold_cols = ['H_T', 'H_V', 'H_TOT']
bold_cols = [col for col in dfH.columns if col.startswith(tuple([x + ' ' for x in bold_cols]))]

dfH.style \
    .format("{:.1f}") \
    .set_properties(subset=bold_cols, **{"font-weight": "bold"}) \
    .set_caption("Total Heat Tranfer Coefficients by zone [W/K]") \
    .set_table_styles(styles)

In [ ]:
bold_cols = ['PHI_T', 'PHI_V', 'PHI_TOT', 'PHI_HL']
bold_cols = [col for col in dfPHI.columns if col.startswith(tuple([x + ' ' for x in bold_cols]))]

dfPHI.style \
    .format("{:.0f}") \
    .set_properties(subset=bold_cols, **{"font-weight": "bold"}) \
    .set_caption("Emission and production Heat losses and Loads by zone [W]") \
    .set_table_styles(styles)

In [ ]:
dffv.style \
    .format("{:.0f}") \
    .set_caption("Emission Heat losses and loads by zone and by facade [W]") \
    .set_table_styles(styles)

In [ ]:
dffPHI.style \
    .format("{:.0f}") \
    .set_caption("Emission Heat losses and loads by facade [W]") \
    .set_table_styles(styles)

In [ ]:
lst_cols = ['PHI_T_wl', 'PHI_T_wd', 'PHI_T_dr', 'PHI_V_su', 'PHI_V_ie', 'PHI_RH']
cols = [col for col in dfPHI.columns if col.startswith(tuple([x + ' ' for x in lst_cols]))]
df_plot = dfPHI[cols].copy()
df_plot.index.name = ''

red       = (0.984313725490196, 0.5019607843137255, 0.4470588235294118) 
orange    = (0.9921568627450981, 0.7058823529411765, 0.3843137254901961)  
yellow    = (1.0, 0.9294117647058824, 0.43529411764705883)
blue      = (0.09019607843137255, 0.7450980392156863, 0.8117647058823529)
lightblue = (0.6196078431372549, 0.8549019607843137, 0.8980392156862745)  
grey      = (0.83, 0.83, 0.83) 

df_plot.loc[::-1].plot.barh(figsize=(8,3), stacked=True, color = (red, orange, yellow, blue, lightblue, grey))
plt.xlabel("Heating Power [W]")
plt.title('Breakdown of the zone Heat Loads [W]')
plt.grid(linestyle = 'dotted');

In [ ]:
with pd.ExcelWriter(project + "_output.xlsx") as writer: 

    # DATA
    
    # General data
    df_gen_data.to_excel(writer, sheet_name='general_data')

    # Reheat data
    df_rh_data.to_excel(writer, sheet_name='reheat_data') 

    # Ventilation data
    dfv.to_excel(writer, sheet_name='ventilation_data') 

    # Materials data
    materials.to_excel(writer, sheet_name='materials_data')

    # Wall types layers data
    wt_layers.to_excel(writer, sheet_name='wall_types_layers_data')

    # Wall types environmental data
    wt_env.to_excel(writer, sheet_name='wall_types_env_data')

    # Walls data
    wl.to_excel(writer, sheet_name='wall_data')

    # Windows data
    wd.to_excel(writer, sheet_name='windows_data')

    # Doors data
    dr.to_excel(writer, sheet_name='doors_data')
    

    # RESULTS

    # dfwt : Walls thicknesses [m] and conductive resistance [m²K/W]
    dfwt.round(2).to_excel(writer, sheet_name='walls_R') 
    
    # dfAU  : Walls U_values [W/m²K] - Temperature Factors - Areas [m²] - Heat Tranfer Coefficients [W/K]
    dfAU.round(2).to_excel(writer, sheet_name='walls_AU') 

    # dfgrAU : Walls Areas [m²] and Heat Tranfer Coefficients [W/K] by zone and wall type
    dfgrAU.round(2).round(2).to_excel(writer, sheet_name='walls_AU_grouped') 

    # dfAHT : Wall Areas [m²] and Heat Tranfer Coefficients [W/K] by zone
    dfAHT.round(2).to_excel(writer, sheet_name='HT_by_zone') 

    # dfH   : Total Heat Tranfer Coefficients by zone [W/K]
    dfH.round(2).to_excel(writer, sheet_name='HTOT_by_zone') 

    # dfPHI : Emission and production Heat losses and Loads by zone [W]
    dfPHI.astype(int).to_excel(writer, sheet_name='HL_by_zone') 

    # dffv  : Emission Heat losses and loads by zone and by facade [W]
    dffv.astype(int).to_excel(writer, sheet_name='HL_by_zone_by_facade') 

    # dffPHI : Emission Heat losses and loads by facade [W]
    dffPHI.astype(int).to_excel(writer, sheet_name='HL_by_facade') 